In [1]:
import ipywidgets as widgets
import sqlite3
import pulp
from IPython.display import display, HTML
import pandas as pd

# Scoring Function #

In [2]:
def load_vote_counts(db_path: str = "nba_players.db") -> dict[str, int]:
    """Load how many times each player was approved across all teams."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT player_full_name, COUNT(*) AS vote_count
        FROM votes
        GROUP BY player_full_name
    """)
    rows = cursor.fetchall()
    conn.close()
    return {name: count for name, count in rows}

# Denial Constraints #

In [3]:
def get_low_3p_players(db_path, threshold=0.20):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(FG3_PCT) AS best_3p
            FROM player_career_stats
            WHERE FG3_PCT IS NOT NULL AND FG3A > 0
            GROUP BY FULL_NAME
        ) WHERE best_3p < ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

def get_high_tov_players(db_path, threshold=260):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(TOV) AS max_tov
            FROM player_career_stats
            WHERE TOV IS NOT NULL
            GROUP BY FULL_NAME
        ) WHERE max_tov > ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

def get_low_ft_players(db_path, threshold=0.65):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(FT_PCT) AS best_ft
            FROM player_career_stats
            WHERE FT_PCT IS NOT NULL AND FTA > 0
            GROUP BY FULL_NAME
        ) WHERE best_ft < ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

# TGD Constraints #

In [4]:
def get_good_scorers(db_path, threshold=1200):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(PTS) AS best_pts
            FROM player_career_stats
            GROUP BY FULL_NAME
        ) WHERE best_pts > ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

def get_good_rebounders(db_path, threshold=500):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(REB) AS best_reb
            FROM player_career_stats
            GROUP BY FULL_NAME
        ) WHERE best_reb > ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

def get_good_defenders(db_path, stl_threshold=80, blk_threshold=80):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(STL) AS best_stl, MAX(BLK) AS best_blk
            FROM player_career_stats
            GROUP BY FULL_NAME
        ) WHERE best_stl > ? OR best_blk > ?
    """, (stl_threshold, blk_threshold))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

def get_good_3p_shooters(db_path, threshold=0.25):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT FULL_NAME FROM (
            SELECT FULL_NAME, MAX(FG3_PCT) AS best_3p
            FROM player_career_stats
            WHERE FG3_PCT IS NOT NULL AND FG3A > 0
            GROUP BY FULL_NAME
        ) WHERE best_3p >= ?
    """, (threshold,))
    result = [r[0] for r in cur.fetchall()]
    conn.close()
    return result

# MIP Solver #

In [5]:
def solve_committee(k, db_path,
                    dc_low_3p=None, dc_high_tov=None, dc_low_ft=None,
                    tgd_scorer=None, tgd_rebounder=None,
                    tgd_defender=None, tgd_shooter=None):
    vote_counts = load_vote_counts(db_path)
    players = list(vote_counts.keys())

    prob = pulp.LpProblem("best_committee", pulp.LpMaximize)
    x = {p: pulp.LpVariable(f"x_{i}", cat="Binary") for i, p in enumerate(players)}

    prob += pulp.lpSum(vote_counts[p] * x[p] for p in players)
    prob += pulp.lpSum(x[p] for p in players) == k

    if dc_low_3p is not None:
        g = [p for p in get_low_3p_players(db_path, dc_low_3p) if p in x]
        if len(g) > 1:
            prob += pulp.lpSum(x[p] for p in g) <= 1

    if dc_high_tov is not None:
        g = [p for p in get_high_tov_players(db_path, dc_high_tov) if p in x]
        if len(g) > 1:
            prob += pulp.lpSum(x[p] for p in g) <= 1

    if dc_low_ft is not None:
        g = [p for p in get_low_ft_players(db_path, dc_low_ft) if p in x]
        if len(g) > 1:
            prob += pulp.lpSum(x[p] for p in g) <= 1

    if tgd_scorer is not None:
        g = [p for p in get_good_scorers(db_path, tgd_scorer) if p in x]
        if g:
            prob += pulp.lpSum(x[p] for p in g) >= 1

    if tgd_rebounder is not None:
        g = [p for p in get_good_rebounders(db_path, tgd_rebounder) if p in x]
        if g:
            prob += pulp.lpSum(x[p] for p in g) >= 1

    if tgd_defender is not None:
        g = [p for p in get_good_defenders(db_path, tgd_defender, tgd_defender) if p in x]
        if g:
            prob += pulp.lpSum(x[p] for p in g) >= 1

    if tgd_shooter is not None:
        g = [p for p in get_good_3p_shooters(db_path, tgd_shooter) if p in x]
        if g:
            prob += pulp.lpSum(x[p] for p in g) >= 1

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    committee = [p for p in players if pulp.value(x[p]) == 1]
    score = int(pulp.value(prob.objective))
    return committee, score


# UI #

In [6]:
style = {"description_width": "initial"}

# Settings -----------------------------
db_dropdown    = widgets.Dropdown(options=["nba_players.db"], description="Database:", style=style)
committee_size = widgets.IntText(value=5, description="Committee Size (k):", style=style)
voting_rule    = widgets.Dropdown(options=["Approval Voting"], description="Voting Rule:", style=style)

# Denial Constraints -----------------------------
dc_low_3p_on    = widgets.Checkbox(value=True, description="No 2 non-shooters (FG3% <)")
dc_low_3p_val   = widgets.FloatText(value=0.20)

dc_high_tov_on  = widgets.Checkbox(value=True, description="No 2 high-turnover players (TOV >)")
dc_high_tov_val = widgets.IntText(value=260)

dc_low_ft_on    = widgets.Checkbox(value=True, description="No 2 poor free-throw shooters (FT% <)")
dc_low_ft_val   = widgets.FloatText(value=0.65)

# TGDs ----------------------------- 
tgd_scorer_on    = widgets.Checkbox(value=True, description="Must include a scorer (PTS >)")
tgd_scorer_val   = widgets.IntText(value=1200)

tgd_rebounder_on  = widgets.Checkbox(value=True, description="Must include a rebounder (REB >)")
tgd_rebounder_val = widgets.IntText(value=500)

tgd_defender_on  = widgets.Checkbox(value=True, description="Must include a defender (STL/BLK >)")
tgd_defender_val = widgets.IntText(value=80)

tgd_shooter_on   = widgets.Checkbox(value=True, description="Must include a 3PT shooter (FG3% >=)")
tgd_shooter_val  = widgets.FloatText(value=0.25)

run_btn = widgets.Button(description="Run", button_style="success")

def build_results_table(committee, db):
    vote_counts = load_vote_counts(db)
    conn = sqlite3.connect(db)
    ph = ",".join("?" * len(committee))
    df = pd.read_sql_query(f"""
        SELECT FULL_NAME AS Player,
               MAX(PTS) AS PTS, MAX(AST) AS AST, MAX(REB) AS REB,
               MAX(STL) AS STL, MAX(BLK) AS BLK,
               ROUND(MAX(FG_PCT), 3) AS "FG%",
               ROUND(MAX(FT_PCT), 3) AS "FT%"
        FROM player_career_stats
        WHERE FULL_NAME IN ({ph})
        GROUP BY FULL_NAME
    """, conn, params=committee)
    conn.close()
    df.insert(1, "Approvals", df["Player"].map(vote_counts).fillna(0).astype(int))
    return df.sort_values("Approvals", ascending=False).reset_index(drop=True)

def on_click(b):
    db = db_dropdown.value
    k  = committee_size.value

    committee, score = solve_committee(
        k, db,
        dc_low_3p     = dc_low_3p_val.value     if dc_low_3p_on.value     else None,
        dc_high_tov   = dc_high_tov_val.value   if dc_high_tov_on.value   else None,
        dc_low_ft     = dc_low_ft_val.value     if dc_low_ft_on.value     else None,
        tgd_scorer    = tgd_scorer_val.value    if tgd_scorer_on.value    else None,
        tgd_rebounder = tgd_rebounder_val.value if tgd_rebounder_on.value else None,
        tgd_defender  = tgd_defender_val.value  if tgd_defender_on.value  else None,
        tgd_shooter   = tgd_shooter_val.value   if tgd_shooter_on.value   else None,
    )

    if not committee:
        print("No valid committee found. Try unchecking some constraints.")
        return

    result_df = build_results_table(committee, db)
    result_df.index += 1

    print(f"Winning Committee -- {voting_rule.value} (k={k})")
    print(f"Total Approval Score: {score}")
    print()
    print(result_df.to_string())

run_btn.on_click(on_click)

# Layout -----------------------------
display(widgets.VBox([
    widgets.HTML("<b>Settings</b>"),
    db_dropdown, committee_size, voting_rule,

    widgets.HTML("<hr><b>Denial Constraints (DC)</b>"),
    widgets.HBox([dc_low_3p_on,   dc_low_3p_val]),
    widgets.HBox([dc_high_tov_on, dc_high_tov_val]),
    widgets.HBox([dc_low_ft_on,   dc_low_ft_val]),

    widgets.HTML("<hr><b>Tuple-Generating Dependencies (TGD)</b>"),
    widgets.HBox([tgd_scorer_on,    tgd_scorer_val]),
    widgets.HBox([tgd_rebounder_on, tgd_rebounder_val]),
    widgets.HBox([tgd_defender_on,  tgd_defender_val]),
    widgets.HBox([tgd_shooter_on,   tgd_shooter_val]),

    widgets.HTML("<hr>"),
    run_btn,
]))

Winning Committee -- Approval Voting (k=5)
Total Approval Score: 114

            Player  Approvals   PTS  AST  REB  STL  BLK   FG%   FT%
1      Karl Malone         27  1944  276  788  110   60  0.51  0.73
2       Larry Bird         25  1676  438  690  120   58  0.49  0.89
3   Michael Jordan         25  2153  376  445  168   60  0.49  0.83
4     LeBron James         23  1857  503  512  107   52  0.51  0.73
5  Hakeem Olajuwon         14  1497  170  764  120  213  0.51  0.70
